# Synchrosqueezed wavelet transform

`encoders.synchrosqueezed_cwt` reassigns CWT energy to the instantaneous
frequency estimated from the phase derivative, so ridges become sharp instead
of smeared across the wavelet's bandwidth.

This notebook shows the three cases the implementation is validated on: a
constant sinusoid, a linear chirp with a known frequency law, and a
two-component signal.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tscv_vision import encoders

FS = 200.0
N = 1024
t = np.arange(N) / FS
freqs = np.linspace(FS / N, FS / 2, 256)


def show(signal, title, truth=None):
    sst = encoders.synchrosqueezed_cwt(signal, fs=FS, frequencies=256)
    cwt = encoders.cwt(signal, np.linspace(1.0, 64.0, 256))
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].plot(t, signal, lw=0.6)
    axes[0].set(title=title, xlabel="time (s)")
    axes[1].imshow(cwt, aspect="auto", origin="lower", extent=(0, t[-1], 0, 255))
    axes[1].set(title="CWT (scale index)", xlabel="time (s)")
    axes[2].imshow(sst, aspect="auto", origin="lower",
                   extent=(0, t[-1], freqs[0], freqs[-1]))
    if truth is not None:
        axes[2].plot(t, truth, "r--", lw=1.0, label="analytic")
        axes[2].legend(loc="upper left")
    axes[2].set(title="synchrosqueezed CWT (Hz)", xlabel="time (s)")
    plt.tight_layout()
    plt.show()

## 1. Constant sinusoid at 20 Hz

The ridge should be a flat line at 20 Hz.

In [ ]:
show(np.sin(2 * np.pi * 20.0 * t), "sin(2pi 20 t)", truth=np.full_like(t, 20.0))

## 2. Linear chirp

`x(t) = sin(2 pi (f0 t + k t^2 / 2))` has instantaneous frequency `f0 + k t`.
The dashed line is that analytic law; the ridge should sit on it.

In [ ]:
f0, k = 10.0, 10.0
chirp = np.sin(2 * np.pi * (f0 * t + 0.5 * k * t**2))
show(chirp, "linear chirp", truth=f0 + k * t)

## 3. Two components

A 15 Hz and a 55 Hz tone, which should appear as two separate ridges.

In [ ]:
show(np.sin(2 * np.pi * 15.0 * t) + np.sin(2 * np.pi * 55.0 * t),
     "15 Hz + 55 Hz")

## Concentration

Synchrosqueezing does not add resolution; it concentrates the energy that the
CWT already computed. `benchmark_time_frequency` quantifies the trade against
runtime and memory.

In [ ]:
from tscv_vision.benchmark import benchmark_time_frequency

for name, metrics in benchmark_time_frequency().items():
    print(f"{name:22s} {metrics['seconds'] * 1000:7.1f} ms  "
          f"{metrics['peak_mib']:6.1f} MiB  "
          f"sparsity {metrics['sparsity']:.3f}  "
          f"concentration {metrics['concentration']:.2e}")